# Data Ingestion (Source → Bronze)

This notebook reads the `table_config` metadata to dynamically ingest source data into the bronze layer. It supports full_load and incremental load strategies, and logs every run to `pipeline_run_log`.

In [0]:
import uuid
from datetime import datetime
from pyspark.sql.functions import current_timestamp, lit

CATALOG = "metadata_framework"
META_SCHEMA = f"{CATALOG}.bronze"

def get_active_table_configs():
    """Read all active table configurations from the metadata table."""
    return (spark.read.table(f"{META_SCHEMA}.table_config")
            .where("is_active = true")
            .collect())

def get_source_info(source_id):
    """Fetch source configuration for a given source_id."""
    return (spark.read.table(f"{META_SCHEMA}.source_config")
            .where(f"source_id = '{source_id}' AND is_active = true")
            .collect()[0])

def log_pipeline_run(run_id, table_id, status, layer, records_read=0, records_written=0, records_rejected=0, error_message=None):
    """Insert a record into pipeline_run_log to track this run."""
    log_df = spark.createDataFrame([(
        run_id, table_id, status, layer,
        records_read, records_written, records_rejected,
        error_message,
        datetime.now(),
        datetime.now() if status in ("success", "failed") else None
    )], schema="run_id STRING, table_id STRING, run_status STRING, run_layer STRING, records_read BIGINT, records_written BIGINT, records_rejected BIGINT, error_message STRING, started_at TIMESTAMP, completed_at TIMESTAMP")
    
    log_df.write.mode("append").saveAsTable(f"{META_SCHEMA}.pipeline_run_log")

print("Ingestion utilities loaded.")

Ingestion utilities loaded.


In [0]:
def ingest_table(config_row):
    """
    Ingest a single table based on its metadata configuration.
    Supports full_load and incremental load types.
    """
    table_id = config_row.table_id
    source_id = config_row.source_id
    source_table = config_row.source_table_name
    target_schema = config_row.target_schema
    target_table = config_row.target_table_name
    load_type = config_row.load_type
    incremental_col = config_row.incremental_column
    primary_keys = config_row.primary_keys
    partition_cols = config_row.partition_columns
    
    run_id = str(uuid.uuid4())
    full_target = f"{CATALOG}.{target_schema}.{target_table}"
    
    # Get source info
    source_info = get_source_info(source_id)
    source_path = source_info.source_path
    source_type = source_info.source_type
    
    print(f"\n{'='*60}")
    print(f"Ingesting: {source_table} → {full_target}")
    print(f"Load type: {load_type}")
    
    log_pipeline_run(run_id, table_id, "started", target_schema)
    
    try:
        # Read source data based on source_type
        if source_type == "csv":
            reader = spark.read.format("csv").option("header", "true").option("inferSchema", "true")
            source_df = reader.load(source_path)
        elif source_type == "json":
            source_df = spark.read.json(source_path)
        elif source_type == "parquet":
            source_df = spark.read.parquet(source_path)
        elif source_type == "delta":
            source_df = spark.read.table(source_path)  # source_path is a fully-qualified table name
        else:
            raise ValueError(f"Unsupported source type: {source_type}")
        
        records_read = source_df.count()
        
        # Add metadata columns
        source_df = source_df.withColumn("_ingested_at", current_timestamp()) \
                             .withColumn("_source_file", lit(source_table))
        
        # Write based on load type
        if load_type == "full_load":
            writer = source_df.write.mode("overwrite")
            if partition_cols:
                writer = writer.partitionBy(partition_cols.split(","))
            writer.saveAsTable(full_target)
            
        elif load_type == "incremental":
            # For incremental: use MERGE if primary keys exist, else append
            if primary_keys:
                merge_condition = " AND ".join([f"t.{pk} = s.{pk}" for pk in primary_keys.split(",")])
                
                # Check if target table exists
                table_exists = spark.catalog.tableExists(full_target)
                if not table_exists:
                    source_df.write.saveAsTable(full_target)
                else:
                    # Build dynamic MERGE
                    pk_list = primary_keys.split(",")
                    source_cols = source_df.columns
                    set_clause = ", ".join([f"t.{c} = s.{c}" for c in source_cols if c not in pk_list])
                    insert_cols = ", ".join(source_cols)
                    insert_vals = ", ".join([f"s.{c}" for c in source_cols])
                    
                    merge_sql = f"""
                    MERGE INTO {full_target} t
                    USING {full_target}_staging s
                    ON {merge_condition}
                    WHEN MATCHED THEN UPDATE SET {set_clause}
                    WHEN NOT MATCHED THEN INSERT ({insert_cols}) VALUES ({insert_vals})
                    """
                    # Create staging table
                    source_df.write.mode("overwrite").saveAsTable(f"{full_target}_staging")
                    spark.sql(merge_sql)
                    spark.sql(f"DROP TABLE IF EXISTS {full_target}_staging")
            else:
                source_df.write.mode("append").saveAsTable(full_target)
        
        records_written = spark.read.table(full_target).count()
        print(f"✅ Success: {records_read} records read, {records_written} total in target")
        
        log_pipeline_run(run_id, table_id, "success", target_schema, records_read, records_written)
        return True
        
    except Exception as e:
        error_msg = str(e)[:500]
        print(f"❌ Failed: {error_msg}")
        log_pipeline_run(run_id, table_id, "failed", target_schema, 0, 0, 0, error_msg)
        return False

print("Ingestion function defined.")

In [0]:
# ============================================================================
# Execute ingestion for all active tables
# ============================================================================
configs = get_active_table_configs()
print(f"Found {len(configs)} active table configuration(s):\n")

success_count = 0
failure_count = 0

for config in configs:
    result = ingest_table(config)
    if result:
        success_count += 1
    else:
        failure_count += 1

print(f"\n{'='*60}")
print(f"Ingestion Summary: {success_count} succeeded, {failure_count} failed")